In [ ]:
import requests
from collections import defaultdict


def get_stats(ids):
    """
    Count the frequency of all adjacent pairs in the list of token IDs.
    """
    counts = defaultdict(int)
    for pair in zip(ids, ids[1:]):
        counts[pair] += 1
    return counts


def merge(ids, pair, new_id):
    """
    Replace all occurrences of a specific pair in the list with a new token ID.
    """
    new_ids = []
    i = 0
    while i < len(ids):
        if i == len(ids) - 1:
            new_ids.append(ids[i])
            break
        if (ids[i], ids[i + 1]) == pair:
            new_ids.append(new_id)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids


class BPE_Tokenizer:
    """
    Byte Pair Encoding (BPE) tokenizer class.
    """

    def __init__(self):
        # Initial vocabulary: 0-255 byte values
        self.vocab = {i: bytes([i]) for i in range(256)}
        self.merges = {}

    def train(self, text, vocab_size=512, verbose=False):
        """
        Train the BPE tokenizer on the given text.

        :param text: Training text
        :param vocab_size: Target vocabulary size (>=256)
        :param verbose: Print merge steps if True
        """
        if vocab_size < 256:
            raise ValueError("Vocabulary size must be at least 256.")

        num_merges = vocab_size - 256
        ids = list(text.encode("utf-8"))

        for i in range(num_merges):
            stats = get_stats(ids)
            if not stats:
                break

            top_pair = max(stats, key=stats.get)
            new_id = 256 + i
            ids = merge(ids, top_pair, new_id)

            self.merges[top_pair] = new_id
            self.vocab[new_id] = self.vocab[top_pair[0]] + self.vocab[top_pair[1]]

            if verbose:
                p0 = self.vocab[top_pair[0]].decode('utf-8', errors='replace')
                p1 = self.vocab[top_pair[1]].decode('utf-8', errors='replace')
                print(f"Merge {i + 1}/{num_merges}: {top_pair} -> {new_id} ('{p0}{p1}')")

    def encode(self, text):
        """
        Encode a string into a list of token IDs using the trained BPE merges.
        """
        ids = list(text.encode("utf-8"))
        while len(ids) >= 2:
            stats = get_stats(ids)
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break
            new_id = self.merges[pair]
            ids = merge(ids, pair, new_id)
        return ids

    def decode(self, ids):
        """
        Decode a list of token IDs back to a string.
        """
        tokens = b"".join(self.vocab[i] for i in ids)
        return tokens.decode("utf-8", errors="replace")


if __name__ == "__main__":
    # Download Tiny Shakespeare corpus
    url = "https://raw.githubusercontent.com/IAmPara0x/fast-bpe/main/tinyshakespeare.txt"
    try:
        response = requests.get(url)
        response.raise_for_status()
        corpus = response.text
        print(f"Successfully downloaded corpus ({len(corpus)} characters).")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading corpus: {e}")
        corpus = ""

    if corpus:
        vocab_size = 512
        tokenizer = BPE_Tokenizer()

        print(f"\nTraining tokenizer with vocab size = {vocab_size}...")
        tokenizer.train(corpus, vocab_size=vocab_size, verbose=True)
        print("\nTraining complete.")

        # Test the tokenizer
        test_text = "ROMEO:\nIs it even so? then I defy you, stars!"
        print(f"\nOriginal Text: '{test_text}'")
        encoded_ids = tokenizer.encode(test_text)
        print(f"Encoded IDs: {encoded_ids}")
        print(f"Number of tokens: {len(encoded_ids)}")
        decoded_text = tokenizer.decode(encoded_ids)
        print(f"Decoded Text: '{decoded_text}'")

        # Verification
        print(f"\nOriginal and decoded texts match: {test_text == decoded_text}")
        original_bytes = len(test_text.encode('utf-8'))
        compressed_tokens = len(encoded_ids)
        print(f"Original text length in bytes: {original_bytes}")
        print(f"Compressed length in tokens: {compressed_tokens}")
        print(f"Compression ratio: {original_bytes / compressed_tokens:.2f}x")

Successfully downloaded corpus (1115394 characters).

Training tokenizer with vocab size = 512...
Merge 1/256: (101, 32) -> 256 ('e ')
Merge 2/256: (116, 104) -> 257 ('th')
Merge 3/256: (116, 32) -> 258 ('t ')
Merge 4/256: (115, 32) -> 259 ('s ')
Merge 5/256: (100, 32) -> 260 ('d ')
Merge 6/256: (44, 32) -> 261 (', ')
Merge 7/256: (111, 117) -> 262 ('ou')
Merge 8/256: (101, 114) -> 263 ('er')
Merge 9/256: (105, 110) -> 264 ('in')
Merge 10/256: (121, 32) -> 265 ('y ')
Merge 11/256: (97, 110) -> 266 ('an')
Merge 12/256: (58, 10) -> 267 (':
')
Merge 13/256: (111, 114) -> 268 ('or')
Merge 14/256: (111, 32) -> 269 ('o ')
Merge 15/256: (101, 110) -> 270 ('en')
Merge 16/256: (10, 10) -> 271 ('

')
Merge 17/256: (97, 114) -> 272 ('ar')
Merge 18/256: (32, 257) -> 273 (' th')
Merge 19/256: (111, 110) -> 274 ('on')
Merge 20/256: (108, 108) -> 275 ('ll')
Merge 21/256: (104, 97) -> 276 ('ha')
Merge 22/256: (44, 10) -> 277 (',
')
Merge 23/256: (46, 271) -> 278 ('.

')
Merge 24/256: (105, 259) -> 279